In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI()

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field


class GrammarRule(BaseModel):
    id: str = Field(description="Unique identifier, for example a1-present-simple-third-person")
    level: Literal["A1", "A2", "B1"] = Field(description="CEFR learner level")
    category: str = Field(description="General category such as tenses, nouns, articles, or conditionals")
    topic: str = Field(description="Specific grammar topic")
    title: str = Field(description="Short descriptive title for the rule")
    rule: str = Field(description="The grammar rule stated clearly and accurately")
    explanation: str = Field(description="Simple explanation suitable for an English learner")
    correct_example_1: str = Field(description="First grammatically correct example")
    correct_example_2: str = Field(description="Second grammatically correct example")
    incorrect_example: str = Field(description="A grammatically incorrect example")
    corrected_example: str = Field(description="Corrected version of the mistake")
    error_reason: str = Field(description="Simple explanation of why the incorrect example is wrong")
    keywords: str = Field(description="Keywords learners may use when asking about this rule")


class GrammarDataset(BaseModel):
    rules: list[GrammarRule]

In [9]:
prompt = """
Generate exactly 30 English grammar rule cards for adult learners
between CEFR levels A1 and B1.

Requirements:

1. Include a balanced mixture of A1, A2, and B1 rules.
2. Focus on common learner mistakes.
3. Each card must cover one specific grammar rule.
4. Do not create duplicate or strongly overlapping cards.
5. Keep explanations clear and concise.
6. Provide grammatically correct examples.
7. Use British English conventions.
8. Include topics such as:
   - the verb be
   - subject pronouns
   - articles
   - plurals
   - present simple
   - present continuous
   - past simple
   - future forms
   - countable and uncountable nouns
   - comparatives
   - modal verbs
   - present perfect
   - conditionals
   - passive voice
   - gerunds and infinitives
9. Use lowercase, hyphen-separated IDs.
""".strip()


response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    text_format=GrammarDataset,
)

grammar_dataset = response.output_parsed

print(len(grammar_dataset.rules))

32


In [10]:
import pandas as pd


records = [rule.model_dump() for rule in grammar_dataset.rules]
df = pd.DataFrame(records)
df.to_csv("data/grammar_rules.csv",index=False)

In [11]:
df.head()

,id,level,category,topic,title,rule,explanation,correct_example_1,correct_example_2,incorrect_example,corrected_example,error_reason,keywords
0,a1-be-am-is-are,A1,be,the verb be,Using be in the present simple,"Use am with I, is with he, she and it, and are...",The verb be changes with the subject.,I am tired.,They are from Spain.,She are happy.,She is happy.,"Use is with she, not are.","be, am, is, are, present simple"
1,a1-subject-pronouns,A1,pronouns,subject pronouns,Using subject pronouns correctly,"Use subject pronouns before the verb: I, you, ...",Subject pronouns replace names and come before...,He works in a bank.,We live in Bristol.,Me work in a bank.,I work in a bank.,Me is not a subject pronoun.,"subject pronouns, I, you, he, she, it, we, they"
2,a1-articles-a-an-the,A1,articles,articles,"Using a, an and the","Use a before consonant sounds, an before vowel...",Articles tell us if a noun is general or speci...,I have a car.,She wants an apple.,He bought the book for the first time.,He bought a book for the first time.,Use a when the noun is not specific.,"articles, a, an, the"
3,a1-plurals-regular,A1,nouns,plurals,Regular plural nouns,Most regular nouns make the plural with -s or ...,"Add -s to most nouns, and -es after endings li...",There are two cats.,She has three boxes.,I saw two dog.,I saw two dogs.,Plural nouns usually need -s or -es.,"plurals, singular, plural, -s, -es"
4,a1-present-simple-routines,A1,present simple,present simple,Present simple for routines,"Use the present simple for routines, habits an...",We use it for things that happen regularly or ...,I get up at seven.,Water boils at 100 degrees.,I am getting up at seven every day.,I get up at seven every day.,"A regular routine needs the present simple, no...","present simple, routines, habits, facts"


In [13]:
print(df["level"].value_counts())

level
A2    14
A1    10
B1     8
Name: count, dtype: int64


In [ ]:
# checking for duplicate IDs and titles
duplicate_ids = df[df["id"].duplicated(keep=False)]
duplicate_ids

duplicate_titles = df[df["title"].duplicated(keep=False)]
duplicate_titles

,id,level,category,topic,title,rule,explanation,correct_example_1,correct_example_2,incorrect_example,corrected_example,error_reason,keywords


In [19]:
import pandas as pd

df = pd.read_csv("data/grammar_rules.csv")
df = df.fillna("")

documents = df.to_dict(orient="records")

In [20]:
documents[0]

{'id': 'a1-be-am-is-are',
 'level': 'A1',
 'category': 'be',
 'topic': 'the verb be',
 'title': 'Using be in the present simple',
 'rule': 'Use am with I, is with he, she and it, and are with you, we and they.',
 'explanation': 'The verb be changes with the subject.',
 'correct_example_1': 'I am tired.',
 'correct_example_2': 'They are from Spain.',
 'incorrect_example': 'She are happy.',
 'corrected_example': 'She is happy.',
 'error_reason': 'Use is with she, not are.',
 'keywords': 'be, am, is, are, present simple'}

In [22]:
from minsearch import Index

index = Index(
    text_fields=[
        "title",
        "rule",
        "explanation",
        "correct_example_1",
        "correct_example_2",
        "incorrect_example",
        "corrected_example",
        "error_reason",
        "keywords",
    ],
    keyword_fields=[
        "id",
        "level",
        "category",
        "topic",
    ],
)

index.fit(documents)

In [23]:
def search_grammar_rules(
    query: str,
    level: str | None = None,
    num_results: int = 5,
):
    filter_dict = {}

    if level:
        filter_dict["level"] = level

    boost_dict = {
        "title": 3.0,
        "keywords": 2.5,
        "rule": 2.0,
        "incorrect_example": 2.0,
        "error_reason": 1.5,
        "explanation": 1.0,
    }

    results = index.search(
        query=query,
        filter_dict=filter_dict,
        boost_dict=boost_dict,
        num_results=num_results,
    )

    return results


In [24]:
results = search_grammar_rules(
    query="Why is 'she go to school' incorrect?",
    level="A1",
)

for result in results:
    print(result["id"])
    print(result["title"])
    print(result["rule"])
    print()

a1-be-am-is-are
Using be in the present simple
Use am with I, is with he, she and it, and are with you, we and they.

a1-possessive-adjectives
Possessive adjectives before nouns
Use my, your, his, her, its, our and their before a noun.

a1-subject-pronouns
Using subject pronouns correctly
Use subject pronouns before the verb: I, you, he, she, it, we, they.

a1-present-simple-third-person
Third person singular in the present simple
In the present simple, add -s or -es to the verb with he, she or it.

a1-present-simple-dont-doesnt
Present simple negatives with do not and does not
Use do not/does not + base verb in negatives.



Additional testing for relevance

In [25]:
results = search_grammar_rules(
    query="When should I use a or an?",
    level="A1",
)

for result in results:
    print(result["title"])

Using a, an and the
Regular plural nouns
Third person singular in the present simple
Present simple questions with do and does
Present simple for routines


In [26]:
results = search_grammar_rules(
    query="What is the difference between past simple and present perfect?",
    level="B1",
)

for result in results:
    print(result["title"])

Passive voice with be + past participle
Already and yet with the present perfect
Present perfect with since and for
Using by in passive sentences
First conditional for real future possibilities


In [27]:
prompt_template = """
You are a patient English grammar tutor for adult learners.

Answer the QUESTION using only the grammar information in the CONTEXT.

Rules:

1. Do not invent grammar rules that are absent from the context.
2. Use language appropriate for the learner's stated level.
3. When correcting a sentence, include:
   - the corrected sentence
   - what changed
   - the relevant grammar rule
   - one short practice question
4. Keep the explanation concise.
5. Use British English.
6. When the context does not contain enough information, say that the
   knowledge base does not contain a suitable rule.

LEARNER LEVEL: {level}

QUESTION:
{question}

CONTEXT:
{context}
""".strip()


entry_template = """
Rule ID: {id}
Level: {level}
Category: {category}
Topic: {topic}
Title: {title}
Rule: {rule}
Explanation: {explanation}
Correct example 1: {correct_example_1}
Correct example 2: {correct_example_2}
Incorrect example: {incorrect_example}
Corrected example: {corrected_example}
Reason: {error_reason}
""".strip()

In [36]:
def build_prompt(
    query: str,
    search_results: list[dict],
    level: str,
):
    context_entries = []

    for document in search_results:
        entry = entry_template.format(**document)
        context_entries.append(entry)

    context = "\n\n".join(context_entries)

    prompt = prompt_template.format(
        question=query,
        level=level,
        context=context,
    )

    return prompt

In [37]:
results = search_grammar_rules(
    "Why is 'she go to work' wrong?",
    level="A1",
)

prompt = build_prompt(
    query="Why is 'she go to work' wrong?",
    search_results=results,
    level="A1",
)

print(prompt)

You are a patient English grammar tutor for adult learners.

Answer the QUESTION using only the grammar information in the CONTEXT.

Rules:

1. Do not invent grammar rules that are absent from the context.
2. Use language appropriate for the learner's stated level.
3. When correcting a sentence, include:
   - the corrected sentence
   - what changed
   - the relevant grammar rule
   - one short practice question
4. Keep the explanation concise.
5. Use British English.
6. When the context does not contain enough information, say that the
   knowledge base does not contain a suitable rule.

LEARNER LEVEL: A1

QUESTION:
Why is 'she go to work' wrong?

CONTEXT:
Rule ID: a1-be-am-is-are
Level: A1
Category: be
Topic: the verb be
Title: Using be in the present simple
Rule: Use am with I, is with he, she and it, and are with you, we and they.
Explanation: The verb be changes with the subject.
Correct example 1: I am tired.
Correct example 2: They are from Spain.
Incorrect example: She are happ

In [38]:
def call_llm(
    prompt: str,
    model: str = "gpt-5.4-mini",
):
    response = openai_client.responses.create(
        model=model,
        input=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    return response.output_text

Making a RAG function by combining the functions above

In [39]:
def rag(
    query: str,
    level: str = "A1",
    model: str = "gpt-5.4-mini",
):
    search_results = search_grammar_rules(
        query=query,
        level=level,
        num_results=5,
    )

    prompt = build_prompt(
        query=query,
        search_results=search_results,
        level=level,
    )

    answer = call_llm(
        prompt=prompt,
        model=model,
    )

    source_ids = [
        document["id"]
        for document in search_results
    ]

    return {
        "answer": answer,
        "source_ids": source_ids,
    }

In [40]:
result = rag(
    query="Why is 'She go to school every day' incorrect?",
    level="A1",
)

print(result["answer"])
print()
print("Retrieved rules:", result["source_ids"])

**Corrected sentence:** **She goes to school every day.**

**What changed:**  
- **go** → **goes**

**Relevant grammar rule:**  
In the present simple, add **-s** or **-es** to the verb with **he, she,** or **it**.

**Why the original is incorrect:**  
**She** is third person singular, so the verb must change to **goes**.

**Practice question:**  
Complete the sentence: **He ___ to work every day.**

Retrieved rules: ['a1-be-am-is-are', 'a1-present-simple-routines', 'a1-possessive-adjectives', 'a1-subject-pronouns', 'a1-present-simple-third-person']


In [41]:
result = rag(
    query="Please explain when to use a and an.",
    level="A1",
)

print(result["answer"])

Use **a** before **consonant sounds** and **an** before **vowel sounds**.

- **a car**  
- **an apple**

**Rule:** Articles tell us if a noun is general or specific. Use **a** when the noun is **not specific**.

**Practice question:**  
Should we say **a** or **an** in **___ book**?


In [42]:
result = rag(
    query="I have visited London yesterday. Is this sentence correct?",
    level="B1",
)

print(result["answer"])

This sentence is **not correct**.

**Correct sentence:**  
**I visited London yesterday.**

**What changed:**  
- **have visited** → **visited**

**Relevant grammar rule:**  
The knowledge base does not contain a suitable rule for **present perfect with a finished time expression like “yesterday”**.

**Practice question:**  
Choose the correct sentence:  
1. I have seen her yesterday.  
2. I saw her yesterday.
